In [ ]:
import os
os.environ["QT_PLUGIN_PATH"] = ""
os.environ["QT_QPA_PLATFORM_PLUGIN_PATH"] = ""
from PyQt5.QtCore import QLibraryInfo, QLibraryInfo
print("Qt path:", QLibraryInfo.location(QLibraryInfo.PrefixPath))

In [ ]:
import sys,os
sys.path.append('../../package/')
%run ../../package/init_jupyter.py

In [ ]:
%gui qt
import sys
from PyQt5.QtWidgets import QApplication,QStyleFactory
print(QStyleFactory.keys())

app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

try:
    app.setStyle('Fusion')
except Exception as e:
    print(f"Failed to set style: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mujoco as mj
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import clear_output, display
from scipy.spatial import distance, procrustes
import json
sys.path.append('../')
sys.path.append('../../asset/')
sys.path.append('../../package/') # for 'preretarget'
sys.path.append('../../package/bvh_helper/') # for 'bvh_parser'
sys.path.append('../../package/yoon/')
sys.path.append('../../package/mujoco_helper/')
sys.path.append('../../package/motion_retarget')
sys.path.append('../../package/kinematics_helper/')
sys.path.append('../../package/utility/')
sys.path.append('../../package/smpl/')
sys.path.append('../../package/irvae_tools/')
from bvh_parser import * # for cmu mocap parsing
from motion_retarget import *
from geometry import relaxed_distortion_measure
from clustering_robots import *
from _robot_info import *
from mujoco_parser import *
from utils import *
from smpl_model import *
from smpl_parser import *

plt.rc('xtick', labelsize=6)
plt.rc('ytick', labelsize=6)
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
print ("PyTorch:[%s]"%(th.__version__))
print("MuJoCo:[%s]" % mj.__version__)

### Load Robot Model

In [ ]:
# Load Humanoid
xml_path = '../../asset/lego_hard/scene_lego_hard.xml'
env = MuJoCoParserClass(name='',rel_xml_path=xml_path,verbose=False)

In [ ]:
bvh_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
secs,chains = get_chains_from_bvh_cmu_mocap(
    bvh_path         = bvh_path,
    p_rate           = 0.056444*1.0,
    zup              = True,
    R_offset         = rpy2r(np.deg2rad([0,0,90])),
    plot_chain_graph = True,
    verbose          = False,
)
secs,chains = secs[1:],chains[1:] # exlude the first frame (T-pose)
L = len(chains); dt = secs[1]-secs[0]; HZ = int(1/dt)
print ("[%s] loaded. L:[%d] dt:[%.4f]sec HZ:[%d]"%(bvh_path,L,dt,HZ))

In [ ]:
# joint_names_for_ik      = [
#     'torso_yaw_joint','torso_roll_joint',
#     'l_shoulder_pitch_joint','l_shoulder_roll_joint','l_shoulder_yaw_joint',
#     'l_elbow_pitch_joint','l_elbow_yaw_joint',
#     'l_wrist_roll_joint','l_wrist_yaw_joint',

#     'r_shoulder_pitch_joint','r_shoulder_roll_joint','r_shoulder_yaw_joint',
#     'r_elbow_pitch_joint','r_elbow_yaw_joint',
#     'r_wrist_roll_joint','r_wrist_yaw_joint',

#     'r_pelvis_pitch_joint','r_knee_pitch_joint','r_ankle_pitch_joint',
#     'l_pelvis_pitch_joint','l_knee_pitch_joint','l_ankle_pitch_joint']

qpos_dir_list = directional_preretarget_postrigging(
    env,
    secs,
    chains,
    # ik_p_spine=False,
    # ik_p_neck=ik_p_neck,
    # leg_fixed=leg_fixed,
    src_name           = 'cmu',
    trgt_name          = 'lego_hard',
    joint_names_for_ik = None,
    max_ik_tick        = 100, # maximum IK iteration
    ik_stepsize        = 1, # IK stepsize
    ik_err_th          = 1e-3, # IK error threshold
    print_ik_failure   = False,
    ik_R_hand          = False,
    ik_R_foot          = False,
    reset_env          = True,
    animate            = False, # animate pre-retarget process
    plot_joi_src       = False,
    plot_joi_trgt      = False,
    render_every       = 1,
    verbose            = True,
    print_every        = 30,
    pause_each_tick    = False,
    post_rigging       = True,
    override           = None,
)

In [ ]:
qpos_dir_list[247]

In [ ]:
import mediapy
env.init_viewer(title='Pre-Retargeting',width=800,height=800)
env.set_viewer(azimuth=169.57500000000022,distance=3.4430763972682676,elevation=-12.974999999999993,lookat=[ 0.02, -0.03,  0.9 ])
# env.set_viewer(azimuth=150.4500000000003,distance=5.925655229922415,elevation=-20.849999999999973,lookat=[ 0.9 ,  1.66, -0.81])
env.reset()
SAVE_VIDEO = True
video = []
tick = 0

env.viewer_pause()
# env.plot_traj(traj=base_dir_traj,rgba=(0,0,0,1),
#                 plot_line=False,plot_cylinder=True,cylinder_r=0.005)
# env.plot_traj(traj=rf_dir_traj,rgba=(1,0,0,1),
#                 plot_line=False,plot_cylinder=True,cylinder_r=0.005)
# env.plot_traj(traj=lf_dir_traj,rgba=(0,0,1,1),
#                 plot_line=False,plot_cylinder=True,cylinder_r=0.005)
while tick < L and env.is_viewer_alive():
# while tick < L//2 and env.is_viewer_alive():
    q_robot = qpos_dir_list[tick]
    env.forward(q=q_robot)
    # env.plot_T(p=np.zeros(3),R=np.eye(3),label="Tick: [%d/%d]"%(tick,L),axis_len=0.5,axis_width=0.005)
#     env.plot_contact_info(
#             r_arrow             = 0.008,
#             h_arrow             = 0.5,
#     )
    env.render()
    video.append(env.grab_image())
    tick += 1
    if tick == L:
        if not SAVE_VIDEO: tick = 0
env.close_viewer()
if SAVE_VIDEO:mediapy.show_video(images=video,fps=60,width=800)

In [ ]:
import math
from pathlib import Path
import cv2
from PIL import Image
from IPython.display import display
_BACKENDS = {
    "auto": None,
    "ffmpeg": getattr(cv2, "CAP_FFMPEG", None),
    "gstreamer": getattr(cv2, "CAP_GSTREAMER", None),
    "msmf": getattr(cv2, "CAP_MSMF", None),
    "avfoundation": getattr(cv2, "CAP_AVFOUNDATION", None),
}
def display_frame_at(video_path, sec, resize_to=None, save_path=None,
                     method="nearest", backend="auto", tolerance_ms=None,
                     sequential_fallback=True):
    """
    Show the frame at a given time (sec) (returns PIL). Falls back to a sequential search if seeking fails.
    method: 'nearest' | 'floor' | 'ceil'
    backend: 'auto' | 'ffmpeg' | 'gstreamer' | 'msmf' | 'avfoundation'
    tolerance_ms: tolerance (ms) for deciding that seeking succeeded. If None, set to 1/fps*1.5.
    """
    be_id = _BACKENDS.get(backend, None)
    cap = cv2.VideoCapture(str(video_path)) if be_id is None else cv2.VideoCapture(str(video_path), be_id)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")
    try:
        backend_name = cap.getBackendName() if hasattr(cap, "getBackendName") else "unknown"
    except Exception:
        backend_name = "unknown"
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = (frame_count / fps) if (fps > 0 and frame_count > 0) else None
    if fps <= 0:
        # some backends return fps as 0. The MSEC/sequential fallback still handles it.
        fps = 30.0  # approximate
    if tolerance_ms is None:
        tolerance_ms = (1000.0 / fps) * 1.5
    # clamp time
    if duration is not None:
        sec = max(0.0, min(float(sec), max(0.0, duration - 1.0 / fps)))
    else:
        sec = max(0.0, float(sec))
    target_ms = sec * 1000.0
    frame = None
    def _ok_enough(current_ms):
        return abs((current_ms or 0.0) - target_ms) <= tolerance_ms
    # 1) try by frame index
    if frame_count > 0:
        idx = int({
            "floor": math.floor(sec * fps),
            "ceil":  math.ceil(sec * fps),
        }.get(method, round(sec * fps)))
        idx = max(0, min(idx, max(0, frame_count - 1)))
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if ok and f is not None:
            pos_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            if _ok_enough(pos_ms):
                frame = f
    # 2) try by time (ms)
    if frame is None:
        cap.set(cv2.CAP_PROP_POS_MSEC, target_ms)
        ok, f = cap.read()
        if ok and f is not None:
            pos_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            if _ok_enough(pos_ms):
                frame = f
    # 3) jump roughly by ratio, then search sequentially
    if frame is None and sequential_fallback:
        if duration:
            cap.set(cv2.CAP_PROP_POS_AVI_RATIO, max(0.0, min(0.9999, sec / duration)))
        # sequential read (exact). Choose nearest/floor/ceil by comparing the previous/current frame
        prev_f, prev_ms = None, None
        while True:
            ok, f = cap.read()
            if not ok or f is None:
                break
            cur_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            if method == "floor":
                if cur_ms >= target_ms:
                    frame = prev_f if prev_f is not None else f
                    break
            elif method == "ceil":
                if cur_ms >= target_ms:
                    frame = f
                    break
            else:  # nearest
                if cur_ms >= target_ms:
                    if prev_f is None:
                        frame = f
                    else:
                        frame = prev_f if abs(prev_ms - target_ms) <= abs(cur_ms - target_ms) else f
                    break
            prev_f, prev_ms = f, cur_ms
        # if nothing was picked even at the end, use the last valid frame
        if frame is None:
            frame = prev_f
    if frame is None:
        cap.release()
        raise RuntimeError(
            f"Seeking failed (backend={backend_name}). Force the FFmpeg backend (keep sequential_fallback=True) "
            f"Consider re-encoding with a different codec."
        )
    # BGR→RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # resize
    if resize_to is not None:
        w, h = resize_to
        frame_rgb = cv2.resize(frame_rgb, (w, h), interpolation=cv2.INTER_AREA)
    img = Image.fromarray(frame_rgb)
    # save options
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        img.save(str(save_path))
    # show
    display(img)
    cap.release()
    return img, {"backend": backend_name, "fps": fps, "frame_count": frame_count, "duration_sec": duration}
# frame_sec=0.6
# video_path = '141_16.mp4'
# display_frame_at(video_path, frame_sec, resize_to=None, save_path=None, method='floor')

In [ ]:
def display_frame_at(video_path, sec=None, frame=None, resize_to=None, save_path=None,
                     method="nearest", backend="auto", tolerance=None,
                     sequential_fallback=True):
    """
    Show the frame at a given time (sec) or a given frame index (frame) (returns PIL).
    - frame: int or float. If float, rounded with method ('nearest'|'floor'|'ceil').
             negative values allowed (-1 = last frame). Takes priority over sec if given.
    - sec: float (seconds). If given, nearest/floor/ceil is chosen by method.
    - tolerance: allowed frame error in frame mode (integer, default=±1),
                 in sec mode, the error in ms (default=1.5/fps).
    - backend: 'auto' | 'ffmpeg' | 'gstreamer' | 'msmf' | 'avfoundation'
    - sequential_fallback: fall back to sequential reading when seeking fails.
    Returns: (PIL.Image, meta dict)
    """
    be_id = _BACKENDS.get(backend, None)
    cap = cv2.VideoCapture(str(video_path)) if be_id is None else cv2.VideoCapture(str(video_path), be_id)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")
    try:
        backend_name = cap.getBackendName() if hasattr(cap, "getBackendName") else "unknown"
    except Exception:
        backend_name = "unknown"

    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = (frame_count / fps) if (fps > 0 and frame_count > 0) else None
    if fps <= 0:
        fps = 30.0  # defensive default for some backends

    img_bgr = None

    # --------------------------
    # 1) frame-index mode
    # --------------------------
    if frame is not None:
        # to avoid 0/1-based confusion: input is assumed 0-based (cast to int)
        if isinstance(frame, float):
            idx = {
                "floor": math.floor,
                "ceil":  math.ceil,
            }.get(method, round)(frame)
        else:
            idx = int(frame)

        # negative index -> count from the end
        if idx < 0 and frame_count > 0:
            idx = frame_count + idx

        # clamp to range
        if frame_count > 0:
            idx = max(0, min(idx, frame_count - 1))
        else:
            idx = max(0, idx)

        # allowed frame error
        tol_frames = 1 if (tolerance is None) else int(max(0, tolerance))

        # (a) seek directly by frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        got = False
        if ok and f is not None:
            # OpenCV returns the index of the "next frame" after a read, so -1
            pos_frame_after = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
            pos_frame = max(0, pos_frame_after - 1)
            if abs(pos_frame - idx) <= tol_frames:
                img_bgr = f
                got = True

        # (b) on failure: sequential scan nearby
        if not got and sequential_fallback:
            start = max(0, idx - 2*max(1, tol_frames))
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            prev_f, prev_i = None, None
            cur_i = start
            while True:
                ok, f = cap.read()
                if not ok or f is None:
                    break
                if cur_i >= idx:
                    if method == "floor":
                        img_bgr = prev_f if prev_f is not None else f
                    elif method == "ceil":
                        img_bgr = f
                    else:  # nearest
                        if prev_f is None:
                            img_bgr = f
                        else:
                            img_bgr = prev_f if abs(idx - (cur_i-1)) <= abs(cur_i - idx) else f
                    got = True
                    break
                prev_f, prev_i = f, cur_i
                cur_i += 1
            if not got:
                img_bgr = prev_f  # fall back to at least the last valid frame

        if img_bgr is None:
            cap.release()
            raise RuntimeError(
                f"Seeking by frame index failed (idx={idx}, backend={backend_name}). "
                f"Consider forcing the FFmpeg backend or re-encoding."
            )

    # --------------------------
    # 2) sec (seconds) mode (original behavior)
    # --------------------------
    else:
        if tolerance is None:
            tolerance = (1000.0 / fps) * 1.5  # ms
        if duration is not None:
            sec = max(0.0, min(float(sec), max(0.0, duration - 1.0 / fps)))
        else:
            sec = max(0.0, float(sec))

        target_ms = sec * 1000.0

        def _ok_enough(current_ms):
            return abs((current_ms or 0.0) - target_ms) <= tolerance

        # 2-1) try index
        if frame_count > 0:
            idx = int({
                "floor": math.floor(sec * fps),
                "ceil":  math.ceil(sec * fps),
            }.get(method, round(sec * fps)))
            idx = max(0, min(idx, max(0, frame_count - 1)))
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, f = cap.read()
            if ok and f is not None:
                pos_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
                if _ok_enough(pos_ms):
                    img_bgr = f

        # 2-2) try ms
        if img_bgr is None:
            cap.set(cv2.CAP_PROP_POS_MSEC, target_ms)
            ok, f = cap.read()
            if ok and f is not None:
                pos_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
                if _ok_enough(pos_ms):
                    img_bgr = f

        # 2-3) jump by ratio, then search sequentially
        if img_bgr is None and sequential_fallback:
            if duration:
                cap.set(cv2.CAP_PROP_POS_AVI_RATIO, max(0.0, min(0.9999, sec / duration)))
            prev_f, prev_ms = None, None
            while True:
                ok, f = cap.read()
                if not ok or f is None:
                    break
                cur_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
                if method == "floor":
                    if cur_ms >= target_ms:
                        img_bgr = prev_f if prev_f is not None else f
                        break
                elif method == "ceil":
                    if cur_ms >= target_ms:
                        img_bgr = f
                        break
                else:  # nearest
                    if cur_ms >= target_ms:
                        if prev_f is None:
                            img_bgr = f
                        else:
                            img_bgr = prev_f if abs(prev_ms - target_ms) <= abs(cur_ms - target_ms) else f
                        break
                prev_f, prev_ms = f, cur_ms
            if img_bgr is None:
                img_bgr = prev_f

        if img_bgr is None:
            cap.release()
            raise RuntimeError(
                f"Seeking failed (backend={backend_name}). Force the FFmpeg backend (keep sequential_fallback=True) "
                f"Consider re-encoding with a different codec."
            )

    # --------------------------
    # common post-processing / saving / display
    # --------------------------
    frame_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if resize_to is not None:
        w, h = resize_to
        frame_rgb = cv2.resize(frame_rgb, (w, h), interpolation=cv2.INTER_AREA)
    img = Image.fromarray(frame_rgb)
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        img.save(str(save_path))
    display(img)
    meta = {
        "backend": backend_name,
        "fps": fps,
        "frame_count": frame_count,
        "duration_sec": (frame_count / fps) if (fps > 0 and frame_count > 0) else None,
    }
    cap.release()
    return img, meta

# 1) by frame index (0-based). 180th frame
hard_frame = 192/776
middle_frame = 382/1536
easy_frame = 73/299
num = 485
display_frame_at("18_15.mp4", frame=num*middle_frame)

# # 2) last frame
# display_frame_at("141_16.mp4", frame=-1)

# # 3) original call in seconds, unchanged
# display_frame_at("141_16.mp4", sec=0.6, method="floor")